# E3 — Entrenamiento PoinTr v4 (solo Fantastic Breaks v2, alineacion correcta)

**Modelo:** PoinTr (Yu et al., 2021) — transformer para shape completion.

**Diferencias respecto a v3:**
- **Solo Fantastic Breaks v2** — datos reales con alineacion corregida por Rocio.
  Sin datos sinteticos (cuya alineacion roto/completo era incorrecta).
- **`CENTRAR_EN_ROTO=False`** — los datos ya estan correctamente alineados;
  no centramos para no romper esa alineacion.
- **300 epocas** — dataset muy pequeno (~48 pares train), necesita mas epocas.
- Sin blacklist (no hay sintetico).

**Contexto:** Rocio descubrio que en Fantastic Breaks v1 el fragmento roto no
coincide en posicion con el objeto completo. La v2 corrige este problema.
Este experimento aisle el efecto de tener datos perfectamente alineados.

**Metricas de referencia:**
- PCN v5: CD=0.0630, F-Score=0.0257
- PoinTr v2 (sintetico filtrado, 150 ep): CD=0.0533, F-Score=0.3278

⏱️ **Tiempo estimado:**
- T4: ~1.5-2 horas (300 epocas, ~48 pares train, batch=32)
- A100: ~40-60 minutos

⚠️ **GPU obligatoria.** Menu → Entorno de ejecucion → Cambiar tipo → T4 o A100 GPU

In [ ]:
# ── CELDA 1: Montar Drive ──────────────────────────────────────
from google.colab import drive
import os

if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    print('Drive ya montado.')
else:
    drive.mount('/content/drive')
    print('Drive montado.')

In [ ]:
# ── CELDA 2: Clonar repos + instalar dependencias ──────────────
import os
import subprocess
import time
from getpass import getpass

if not os.path.exists('/content/PoinTr'):
    print('Clonando PoinTr...')
    subprocess.run(['git', 'clone', 'https://github.com/yuxumin/PoinTr',
                    '/content/PoinTr', '--depth=1', '-q'], capture_output=True)
    print('PoinTr clonado.')
else:
    print('[OK] /content/PoinTr ya existe.')

REPO_DIR = '/content/TFM'
if not os.path.exists(REPO_DIR):
    token = getpass('Token GitHub (ghp_...): ')
    repo_url = f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D'
    subprocess.run(['git', 'clone', repo_url, REPO_DIR, '-q'], capture_output=True)
    del token
    print('Repo TFM clonado.')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], capture_output=True)
    print('[OK] /content/TFM ya existe — actualizado.')

os.chdir(REPO_DIR)
subprocess.run(['git', 'checkout', 'raquel/e3', '-q'], capture_output=True)
print(f'Directorio: {os.getcwd()}')

subprocess.run(['pip', 'install', 'timm', 'easydict', 'pyyaml', '--quiet'])
print('[OK] timm, easydict, pyyaml')

CUDA_OK = True
for ext_name, ext_path in [
    ('pointnet2_ops', '/content/PoinTr/extensions/pointnet2_ops_lib'),
    ('chamfer_dist',  '/content/PoinTr/extensions/chamfer_dist'),
]:
    r = subprocess.run(['pip', 'install', '-e', ext_path, '--quiet'],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print(f'[OK] {ext_name} compilado')
    else:
        print(f'[WARN] {ext_name} — se usara fallback PyTorch')
        CUDA_OK = False
print(f'CUDA extensions: {"OK" if CUDA_OK else "fallback PyTorch"}')

In [ ]:
# ── CELDA 3: RUTAS DE DRIVE ────────────────────────────────────
DRIVE   = '/content/drive/MyDrive'
BASE_E3 = f'{DRIVE}/Datos_E2_E3/E3/Raquel'

VERSION = 'v4_pointr_fbv2'

RUTA_FB_V2 = f'{DRIVE}/Datos_E2_E3/General/Fantastik_Break_Procesado_v2'

RUTA_SALIDA_MODELO     = f'{BASE_E3}/modelos/{VERSION}'
RUTA_SALIDA_RESULTADOS = f'{BASE_E3}/resultados/{VERSION}'

print(f'Rutas PoinTr {VERSION}:')
print(f'  fantastic_breaks_v2: {RUTA_FB_V2}')
print(f'  salida modelo      : {RUTA_SALIDA_MODELO}')
print(f'  salida resultados  : {RUTA_SALIDA_RESULTADOS}')

In [ ]:
# ── CELDA 4: Verificar rutas ────────────────────────────────────
from pathlib import Path

p = Path(RUTA_FB_V2)
existe = p.exists()
print(f'  [{"OK" if existe else "FALTA"}] fantastic_breaks_v2: {RUTA_FB_V2}')

if existe:
    n = len(list(p.glob('*_completo.npy')))
    print(f'  Pares disponibles: {n}')
else:
    raise FileNotFoundError(f'Carpeta no encontrada: {RUTA_FB_V2}')

In [ ]:
# ── CELDA 5: Copiar datos desde Drive ──────────────────────────
import subprocess
from pathlib import Path

dst = Path('Datos/fantastic_breaks/procesado_v2')
if dst.exists() and any(dst.glob('*.npy')):
    print(f'[OK] ya existe: {dst} ({len(list(dst.glob("*.npy")))} .npy)')
else:
    dst.mkdir(parents=True, exist_ok=True)
    print('Copiando Fantastik_Break_Procesado_v2...')
    r = subprocess.run(['rsync', '-a', '--no-links', f'{RUTA_FB_V2}/', str(dst)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'[ERROR rsync] {r.stderr[:300]}')
    else:
        n = len(list(dst.glob('*.npy')))
        print(f'listo — {n} archivos .npy copiados.')

In [ ]:
# ── CELDA 6: Verificar GPU ─────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Sin GPU. Ve a Entorno de ejecucion → Cambiar tipo → A100 o T4')

In [ ]:
# ── CELDA 7: ENTRENAR PoinTr v4 (solo FB v2, 300 epocas) ────────

import sys, os, math, time, types, glob as _glob, shutil, random
from pathlib import Path

sys.path.insert(0, '/content/PoinTr')
sys.path.insert(0, '/content/TFM')
os.chdir('/content/TFM')

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from easydict import EasyDict

device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_STR = str(device)
print(f'Dispositivo: {device}')
if device.type == 'cpu':
    print('AVISO: sin GPU. Cambia a T4/A100.')

# ── Limpiar cache PoinTr ─────────────────────────────────────
_pointr_prefixes = ('models', 'utils.registry', 'utils.config', 'utils.logger',
                    'utils.misc', 'extensions', 'datasets', 'tools')
_to_del = [k for k, v in sys.modules.items()
           if (any(k == p or k.startswith(p + '.') for p in _pointr_prefixes)
               or (hasattr(v, '__file__') and v.__file__
                   and '/content/PoinTr' in str(v.__file__)))]
for _k in _to_del: del sys.modules[_k]
print(f'[reset] {len(_to_del)} modulos PoinTr eliminados del cache')

# ── Patch .cuda() → .to(device) ─────────────────────────────
_n_patched = 0
for _fp in (_glob.glob('/content/PoinTr/models/*.py') +
            _glob.glob('/content/PoinTr/models/**/*.py')):
    try:
        with open(_fp, encoding='utf-8') as _f: _src = _f.read()
        _new = _src.replace('.cuda()', f'.to("{DEVICE_STR}")')
        if _new != _src:
            with open(_fp, 'w', encoding='utf-8') as _f: _f.write(_new)
            _n_patched += 1
    except Exception: pass
print(f'[patch] {_n_patched} archivos: .cuda() → .to("{DEVICE_STR}")')

# ── Mocks CUDA ───────────────────────────────────────────────
def _dummy_cls(name):
    return type(name, (nn.Module,), {
        '__init__': lambda self, *a, **kw: super(type(self), self).__init__(),
        'forward':  lambda self, x, *a, **kw: x,
    })

def _force(mod_name, attrs):
    m = types.ModuleType(mod_name)
    for k, v in attrs.items(): setattr(m, k, v)
    sys.modules[mod_name] = m

def _inject(mod_name, attrs):
    if mod_name not in sys.modules: _force(mod_name, attrs)

def _chamfer_raw(a, b):
    dist = torch.cdist(a, b, p=2)
    return dist.min(dim=2).values, dist.min(dim=1).values

class _ChamferL1(nn.Module):
    def forward(self, a, b):
        d1, d2 = _chamfer_raw(a.contiguous(), b.contiguous())
        return (d1.mean() + d2.mean()) / 2

class _ChamferL2(nn.Module):
    def forward(self, a, b):
        d1, d2 = _chamfer_raw(a.contiguous(), b.contiguous())
        return ((d1**2).mean() + (d2**2).mean()) / 2

class _ChamferL1_PM(nn.Module):
    def forward(self, a, b):
        return torch.cdist(a.contiguous(), b.contiguous(), p=2).min(dim=2).values.mean()

_ch_attrs = {'ChamferDistanceL1': _ChamferL1, 'ChamferDistanceL2': _ChamferL2,
             'ChamferDistanceL1_PM': _ChamferL1_PM, 'chamfer_3DDist': _chamfer_raw}
for _n in ['chamfer', 'chamfer_dist', 'extensions.chamfer_dist',
           'chamfer3D', 'chamfer3D.dist_chamfer_3D']:
    _force(_n, _ch_attrs)
print('[OK] Mock chamfer')

if 'pointnet2_ops' not in sys.modules:
    def _fps(xyz, npoint):
        B, N, _ = xyz.shape; dev = xyz.device
        idx = torch.zeros(B, npoint, dtype=torch.int32, device=dev)
        dist = torch.full((B, N), 1e10, device=dev)
        farthest = torch.randint(0, N, (B,), dtype=torch.long, device=dev)
        bi = torch.arange(B, dtype=torch.long, device=dev)
        for i in range(npoint):
            idx[:, i] = farthest.int()
            c = xyz[bi, farthest].unsqueeze(1)
            dist = torch.min(dist, ((xyz - c)**2).sum(-1))
            farthest = dist.max(-1)[1]
        return idx
    def _gather_op(features, idx):
        B, C, N = features.shape; M = idx.shape[1]
        return features.gather(2, idx.long().unsqueeze(1).expand(B, C, M)).contiguous()
    def _ball_query(radius, nsample, xyz, new_xyz):
        dists = torch.cdist(new_xyz.float(), xyz.float())
        srt = dists.argsort(dim=-1); topk = srt[:, :, :nsample]
        return torch.where(dists.gather(2, topk) > radius,
                           srt[:, :, :1].expand_as(topk), topk).int()
    def _grouping_op(features, idx):
        B, C, N = features.shape; S, K = idx.shape[1], idx.shape[2]
        flat = idx.long().view(B, 1, S*K).expand(B, C, S*K)
        return features.gather(2, flat).view(B, C, S, K).contiguous()
    def _three_nn(unknown, known):
        dists = torch.cdist(unknown.float(), known.float())
        dist2, idx = dists.topk(3, dim=-1, largest=False)
        return dist2.float(), idx.int()
    def _three_interp(features, idx, weight):
        B, C, M = features.shape; N = idx.shape[1]
        flat = idx.long().view(B, 1, N*3).expand(B, C, N*3)
        return (features.gather(2, flat).view(B, C, N, 3) * weight.unsqueeze(1)).sum(-1).contiguous()
    _utils = types.ModuleType('pointnet2_ops.pointnet2_utils')
    for k, v in {'furthest_point_sample': _fps, 'gather_operation': _gather_op,
                 'ball_query': _ball_query, 'grouping_operation': _grouping_op,
                 'three_nn': _three_nn, 'three_interpolate': _three_interp}.items():
        setattr(_utils, k, v)
    _pm2 = types.ModuleType('pointnet2_ops'); _pm2.pointnet2_utils = _utils
    sys.modules['pointnet2_ops'] = _pm2; sys.modules['pointnet2_ops.pointnet2_utils'] = _utils
    print('[OK] Mock pointnet2_ops')
else:
    print('[OK] pointnet2_ops ya disponible')

class _KNN(nn.Module):
    def __init__(self, k, transpose_mode=False):
        super().__init__(); self.k = k; self.transpose_mode = transpose_mode
    def forward(self, ref, query):
        if self.transpose_mode:
            dists = torch.cdist(query.float(), ref.float())
            dk, ik = dists.topk(self.k, dim=-1, largest=False)
            return dk, ik
        r = ref.transpose(1,2).contiguous(); q = query.transpose(1,2).contiguous()
        dists = torch.cdist(q.float(), r.float())
        dk, ik = dists.topk(self.k, dim=-1, largest=False)
        return dk.transpose(1,2), ik.transpose(1,2)
_force('knn_cuda', {'KNN': _KNN})
print('[OK] Mock knn_cuda')

_Gridding = _dummy_cls('Gridding'); _GriddingReverse = _dummy_cls('GriddingReverse')
_CubicFS  = _dummy_cls('CubicFeatureSampling'); _GriddingLoss = _dummy_cls('GriddingLoss')
class _EmdModule(nn.Module):
    def forward(self, xyz1, xyz2):
        return (torch.zeros(xyz1.shape[0], device=xyz1.device),
                torch.zeros(xyz1.shape[0], dtype=torch.int32, device=xyz1.device))
for _base, _attrs in [
    ('gridding',               {'Gridding': _Gridding, 'GriddingReverse': _GriddingReverse}),
    ('gridding_loss',          {'GriddingLoss': _GriddingLoss}),
    ('cubic_feature_sampling', {'CubicFeatureSampling': _CubicFS}),
    ('emd',                    {'emd_module': _EmdModule, 'EarthMoverDistance': _EmdModule}),
]:
    for _prefix in ['', 'extensions.']: _inject(_prefix + _base, _attrs)
print('[OK] Mocks gridding / emd')
print()

from extensions.chamfer_dist import ChamferDistanceL1 as _CDL1
_cd_fn = _CDL1()
def chamfer_distance(pred, gt):
    return _cd_fn(pred.contiguous(), gt.contiguous())
print('[OK] Chamfer para training loop')

def subsample_gt(gt, n):
    return gt[:, torch.randperm(gt.size(1), device=gt.device)[:n], :]

# ── Config modelo ─────────────────────────────────────────────
import yaml

cfg_path = '/content/PoinTr/cfgs/PCN_models/PoinTr.yaml'
if not os.path.exists(cfg_path):
    candidates = _glob.glob('/content/PoinTr/cfgs/**/PoinTr.yaml', recursive=True)
    cfg_path = candidates[0] if candidates else None
if cfg_path and os.path.exists(cfg_path):
    with open(cfg_path) as f: raw_cfg = yaml.safe_load(f)
    model_cfg = EasyDict(raw_cfg.get('model', raw_cfg))
else:
    model_cfg = EasyDict({'NAME': 'PoinTr'})
model_cfg.num_pred = 2048; model_cfg.num_query = 128
print(f'Config: num_pred={model_cfg.num_pred}, num_query={model_cfg.num_query}')

try:
    from models.build import build_model_from_cfg
    model = build_model_from_cfg(model_cfg)
    print('[OK] Modelo via build_model_from_cfg')
except Exception as e1:
    try:
        from models.PoinTr import PoinTr
        model = PoinTr(model_cfg)
        print('[OK] Modelo via PoinTr directa')
    except Exception as e2:
        raise RuntimeError(f'No se pudo inicializar PoinTr.\nError 1: {e1}\nError 2: {e2}')

model = model.to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametros: {n_params:,} | Dispositivo: {device}')

if device.type == 'cpu':
    raise SystemExit('Sin GPU — necesitas T4/A100.')

# ── Dataset: solo Fantastic Breaks v2, sin blacklist ─────────
from E3.dataset import construir_pares, ShapeCompletionDataset
import E3.dataset as _ds
_ds.CENTRAR_EN_ROTO = False
print('[OK] CENTRAR_EN_ROTO=False (datos ya alineados por Rocio)')

_todos = construir_pares(['Datos/fantastic_breaks/procesado_v2'])
print(f'Total pares FB v2: {len(_todos)}')

_rng = random.Random(42); _rng.shuffle(_todos); _n = len(_todos)
_n_train = int(0.8 * _n); _n_val = int(0.1 * _n)
_pares_train = _todos[:_n_train]
_pares_val   = _todos[_n_train:_n_train + _n_val]
_pares_test  = _todos[_n_train + _n_val:]
print(f'  Train: {len(_pares_train)} | Val: {len(_pares_val)} | Test: {len(_pares_test)}')

train_loader = DataLoader(ShapeCompletionDataset(_pares_train, augmentar=True),
                          batch_size=32, shuffle=True,  pin_memory=True)
val_loader   = DataLoader(ShapeCompletionDataset(_pares_val,   augmentar=False),
                          batch_size=32, shuffle=False, pin_memory=True)
test_loader  = DataLoader(ShapeCompletionDataset(_pares_test,  augmentar=False),
                          batch_size=32, shuffle=False, pin_memory=True)
print(f'Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')

# ── Hiperparametros y resume ──────────────────────────────────
EPOCHS   = 300; LR = 1e-4; W_COARSE = 0.5
CKPT_DIR = Path('E3/checkpoints_pointr_v4'); CKPT_DIR.mkdir(parents=True, exist_ok=True)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=5e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
mejor_val = math.inf; train_losses = []; val_losses = []; START_EPOCH = 1

def _find_latest_ckpt():
    drive_ckpts = sorted(Path(RUTA_SALIDA_MODELO).glob('epoch_*.pt')) \
                  if Path(RUTA_SALIDA_MODELO).exists() else []
    local_ckpts = sorted(CKPT_DIR.glob('epoch_*.pt'))
    all_ckpts = drive_ckpts + local_ckpts
    if not all_ckpts: return None
    def _ep(p):
        try: return int(p.stem.split('_')[1])
        except: return 0
    return max(all_ckpts, key=_ep)

latest = _find_latest_ckpt()
if latest:
    print(f'\nReanudando desde: {latest}')
    ckpt_data = torch.load(latest, map_location=device, weights_only=False)
    model.load_state_dict(ckpt_data['model_state_dict'])
    optimizer.load_state_dict(ckpt_data['optimizer_state_dict'])
    train_losses = ckpt_data.get('train_losses', [])
    val_losses   = ckpt_data.get('val_losses', [])
    START_EPOCH  = ckpt_data['epoch'] + 1
    mejor_val    = min(val_losses) if val_losses else math.inf
    for _ in range(START_EPOCH - 1): scheduler.step()
    print(f'  Desde epoca {START_EPOCH}/{EPOCHS} | mejor val={mejor_val:.6f}')
else:
    print('\nEntrenamiento desde epoca 1')

if START_EPOCH > EPOCHS:
    print(f'\nEntrenamiento ya completo ({EPOCHS} epocas).')
    drive_best = Path(RUTA_SALIDA_MODELO) / 'best.pt'
    local_best = CKPT_DIR / 'best.pt'
    if not local_best.exists() and drive_best.exists():
        shutil.copy2(drive_best, local_best)
        print(f'[OK] best.pt copiado de Drive a {local_best}')
    elif local_best.exists():
        print(f'[OK] best.pt ya disponible localmente')
    print('Pasa a Celda 9 para evaluar.')
    raise SystemExit('Entrenamiento ya completado.')

def guardar_ckpt(nombre, epoch):
    data = {'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_losses': train_losses, 'val_losses': val_losses,
            'model_cfg': dict(model_cfg)}
    local_path = CKPT_DIR / nombre
    torch.save(data, local_path)
    try:
        Path(RUTA_SALIDA_MODELO).mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_path, Path(RUTA_SALIDA_MODELO) / nombre)
    except Exception as e:
        print(f'  [WARN] No se pudo guardar en Drive: {e}')

print(f'\n{"Epoca":>7}  {"Train":>10}  {"Val":>10}  {"LR":>9}  {"Tiempo":>7}')
print('-' * 52)

for epoch in range(START_EPOCH, EPOCHS + 1):
    t0 = time.time()
    model.train(); total_train = 0.0
    for roto, completo in train_loader:
        roto, completo = roto.to(device), completo.to(device)
        optimizer.zero_grad()
        out    = model(roto)
        coarse = out[0] if isinstance(out, (list, tuple)) else out
        fine   = out[-1] if isinstance(out, (list, tuple)) else out
        gt_c   = subsample_gt(completo, coarse.size(1))
        loss   = chamfer_distance(fine, completo) + W_COARSE * chamfer_distance(coarse, gt_c)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step(); total_train += loss.item()

    model.eval(); total_val = 0.0
    with torch.no_grad():
        for roto, completo in val_loader:
            roto, completo = roto.to(device), completo.to(device)
            out    = model(roto)
            coarse = out[0] if isinstance(out, (list, tuple)) else out
            fine   = out[-1] if isinstance(out, (list, tuple)) else out
            gt_c   = subsample_gt(completo, coarse.size(1))
            total_val += (chamfer_distance(fine, completo) +
                          W_COARSE * chamfer_distance(coarse, gt_c)).item()

    loss_train = total_train / len(train_loader)
    loss_val   = total_val   / len(val_loader)
    elapsed    = time.time() - t0
    train_losses.append(loss_train); val_losses.append(loss_val)
    scheduler.step(); lr_now = scheduler.get_last_lr()[0]

    print(f'{epoch:>4}/{EPOCHS}  {loss_train:>10.6f}  {loss_val:>10.6f}'
          f'  {lr_now:>9.2e}  {elapsed:>6.1f}s')

    if epoch % 10 == 0:
        guardar_ckpt(f'epoch_{epoch:03d}.pt', epoch)

    if loss_val < mejor_val:
        mejor_val = loss_val
        guardar_ckpt('best.pt', epoch)
        print(f'  -> best.pt guardado (val={mejor_val:.6f})')

print(f'\nFin. Mejor val loss: {mejor_val:.6f}')
print(f'Checkpoints en: {CKPT_DIR} y en Drive: {RUTA_SALIDA_MODELO}')

In [ ]:
# ── CELDA 9: Evaluar PoinTr v4 ─────────────────────────────────
import torch, numpy as np, random, sys, os
from pathlib import Path
from torch.utils.data import DataLoader

sys.path.insert(0, '/content/PoinTr')
sys.path.insert(0, '/content/TFM')
os.chdir('/content/TFM')

DRIVE = '/content/drive/MyDrive'
RUTA_SALIDA_RESULTADOS = f'{DRIVE}/Datos_E2_E3/E3/Raquel/resultados/v4_pointr_fbv2'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt_path = 'E3/checkpoints_pointr_v4/best.pt'
if not Path(ckpt_path).exists():
    ckpt_path = f'{DRIVE}/Datos_E2_E3/E3/Raquel/modelos/v4_pointr_fbv2/best.pt'
    print(f'Cargando desde Drive: {ckpt_path}')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
print(f'best.pt — epoca {ckpt["epoch"]}')

from easydict import EasyDict
try:
    from models.build import build_model_from_cfg
    model = build_model_from_cfg(EasyDict(ckpt['model_cfg']))
except Exception:
    from models.PoinTr import PoinTr
    model = PoinTr(EasyDict(ckpt['model_cfg']))
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(device); model.eval()

from E3.dataset import construir_pares, ShapeCompletionDataset
import E3.dataset as _ds
_ds.CENTRAR_EN_ROTO = False

_todos = construir_pares(['Datos/fantastic_breaks/procesado_v2'])
_rng = random.Random(42); _rng.shuffle(_todos); _n = len(_todos)
_n_train = int(0.8 * _n); _n_val = int(0.1 * _n)
_pares_test = _todos[_n_train + _n_val:]
test_loader = DataLoader(ShapeCompletionDataset(_pares_test, augmentar=False),
                         batch_size=32, shuffle=False, pin_memory=True)
print(f'Test: {len(_pares_test)} pares')

def chamfer_eval(pred, gt):
    dist = torch.cdist(pred, gt, p=2)
    return ((dist.min(dim=2).values.mean() + dist.min(dim=1).values.mean()) / 2).item()

def fscore(pred, gt, umbral=0.01):
    dist_pg = torch.cdist(pred, gt, p=2)
    dist_gp = torch.cdist(gt, pred, p=2)
    prec = (dist_pg.min(dim=2).values < umbral).float().mean()
    rec  = (dist_gp.min(dim=2).values < umbral).float().mean()
    if prec + rec < 1e-8: return 0.0
    return (2 * prec * rec / (prec + rec)).item()

cds, fscores = [], []
with torch.no_grad():
    for roto, completo in test_loader:
        roto, completo = roto.to(device), completo.to(device)
        out = model(roto)
        fine = out[-1] if isinstance(out, (list, tuple)) else out
        for i in range(len(roto)):
            p = fine[i:i+1]; g = completo[i:i+1]
            cds.append(chamfer_eval(p, g))
            fscores.append(fscore(p, g))

cd_mean = np.mean(cds); fs_mean = np.mean(fscores)
print(f'\n=== PoinTr v4 (FB v2 alineado) ===')
print(f'  CD-L1   : {cd_mean:.4f}')
print(f'  F-Score : {fs_mean:.4f}')
print(f'  Muestras: {len(cds)}')
print(f'\n  Referencia PCN v5 : CD=0.0630, F=0.0257')
print(f'  Referencia PoinTr v2: CD=0.0533, F=0.3278')

import shutil
resumen_dir = Path('E3/resultados_pointr_v4'); resumen_dir.mkdir(parents=True, exist_ok=True)
with open(resumen_dir / 'resumen.txt', 'w') as f:
    f.write(f'Modelo: PoinTr v4 (solo FB v2 alineado)\nEpoca best: {ckpt["epoch"]}\n')
    f.write(f'CD-L1: {cd_mean:.6f}\nF-Score: {fs_mean:.6f}\nMuestras test: {len(cds)}\n')
Path(RUTA_SALIDA_RESULTADOS).mkdir(parents=True, exist_ok=True)
shutil.copytree('E3/resultados_pointr_v4', RUTA_SALIDA_RESULTADOS, dirs_exist_ok=True)
print(f'Resultados guardados: {RUTA_SALIDA_RESULTADOS}')

In [ ]:
# ── CELDA 10: Curvas de aprendizaje ────────────────────────────
import matplotlib.pyplot as plt, torch
from pathlib import Path

ckpt_path = 'E3/checkpoints_pointr_v4/best.pt'
if not Path(ckpt_path).exists():
    DRIVE = '/content/drive/MyDrive'
    ckpt_path = f'{DRIVE}/Datos_E2_E3/E3/Raquel/modelos/v4_pointr_fbv2/best.pt'
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)

fig, ax = plt.subplots(figsize=(10, 4))
epochs = range(1, len(ckpt['train_losses']) + 1)
ax.plot(epochs, ckpt['train_losses'], label='Train', color='#1565C0', alpha=0.8)
ax.plot(epochs, ckpt['val_losses'],   label='Val',   color='#C62828', alpha=0.8)
ax.axvline(ckpt['epoch'], color='#2E7D32', linestyle='--', alpha=0.7,
           label=f'best (e{ckpt["epoch"]})')
ax.set_xlabel('Epoca')
ax.set_ylabel('Loss (CD-L1 + 0.5*CD-coarse)')
ax.set_title('PoinTr v4 — solo FB v2 alineado, 300 epocas')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig('E3/resultados_pointr_v4/curvas.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Mejor epoch: {ckpt["epoch"]} | val loss: {min(ckpt["val_losses"]):.6f}')